# [[7,1,3]] Steane Code Data Generation - Entanglement Swapping

Self-contained implementation for syndrome data generation using **entanglement swapping** protocol.

**Protocol:**
1. Encode logical qubit at source using [[7,1,3]] Steane code
2. Distribute Bell pairs along path edges (via SWAP through channel qubits)
3. Perform entanglement swapping at intermediate nodes (Bell measurements)
4. Teleport encoded state from source to sink
5. Apply Pauli corrections at sink
6. Measure syndrome at sink

**Fixed Allocation Strategy:**
- Source data qubits: ALWAYS indices 0-6
- Sink ancilla qubits: ALWAYS indices 7-12
- Edge qubits: Fixed per-edge mapping (consistent noise profiles)

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
from qiskit import QuantumCircuit, transpile, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit.visualization import plot_histogram

import numpy as np
import networkx as nx
import pickle
import os
import time
from typing import Dict, List, Tuple
from itertools import combinations

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Experiment parameters
MAX_HOPS = 3
MIN_HOPS = 2
NUM_SHOTS = 4096
OPTIMIZATION_LEVEL = 0
SEED = 1234

# Noise parameters
ERROR_RATE_2Q = 0.01  # 2Q depolarizing error rate

# Paths
NETWORK_GRAPH_PATH = '../networkgraphs/2x3_grid_network_graph.pkl'
OUTPUT_DIR = '../pickle_files/syndrome_data_713_es/'

# FakeFez native gates
BASIS_GATES = ['cz', 'id', 'rz', 'sx', 'x']

# [[7,1,3]] Steane Code parameters
NUM_DATA_QUBITS = 7
NUM_ANCILLA_QUBITS = 6
NUM_SYNDROME_BITS = 6
NUM_SYNDROMES = 64  # 2^6

# Stabilizers (X-type first, then Z-type)
STABILIZERS_713 = [
    "IIIXXXX",  # X on q3, q4, q5, q6
    "IXXIIXX",  # X on q1, q2, q5, q6
    "XIXIXIX",  # X on q0, q2, q4, q6
    "IIIZZZZ",  # Z on q3, q4, q5, q6
    "IZZIIZZ",  # Z on q1, q2, q5, q6
    "ZIZIZIZ",  # Z on q0, q2, q4, q6
]

# Logical qubit index
LOGICAL_QUBIT_INDEX = 0

print(f"Configuration:")
print(f"  Code: [[7,1,3]] Steane")
print(f"  Data qubits: {NUM_DATA_QUBITS}")
print(f"  Ancilla qubits: {NUM_ANCILLA_QUBITS}")
print(f"  Syndrome bits: {NUM_SYNDROME_BITS}")
print(f"  Hops: {MIN_HOPS}-{MAX_HOPS}")
print(f"  Shots: {NUM_SHOTS}")
print(f"  2Q error rate: {ERROR_RATE_2Q}")
print(f"  Network graph: {NETWORK_GRAPH_PATH}")
print(f"  Output directory: {OUTPUT_DIR}")

## Load Network and Build Fixed Allocation

In [ ]:
# =============================================================================
# LOAD NETWORK GRAPH
# =============================================================================

with open(NETWORK_GRAPH_PATH, 'rb') as f:
    network_graph = pickle.load(f)

print(f"Network nodes: {list(network_graph.nodes())}")
print(f"Network edges: {list(network_graph.edges())}")
nx.draw(network_graph, with_labels=True)

In [ ]:
# =============================================================================
# FIXED ALLOCATION MAPPING
# =============================================================================
# Ensures consistent noise profiles across all paths:
# - Source data qubits: ALWAYS indices 0-6
# - Sink ancilla qubits: ALWAYS indices 7-12
# - Each edge (i,j) ALWAYS uses the same qubit range

def get_edge_key(node_a: int, node_b: int) -> Tuple[int, int]:
    """Get canonical edge key (smaller node first)."""
    return (min(node_a, node_b), max(node_a, node_b))

# Get sorted edges
_edges = sorted({get_edge_key(u, v) for u, v in network_graph.edges()})

# Fixed qubit ranges
SOURCE_DATA_QUBITS = list(range(0, NUM_DATA_QUBITS))  # 0-6
SINK_ANCILLA_QUBITS = list(range(NUM_DATA_QUBITS, NUM_DATA_QUBITS + NUM_ANCILLA_QUBITS))  # 7-12

# Edge qubits: 14 Bell pairs + 1 path qubit per edge = 15 qubits
EDGE_STRIDE = 2 * NUM_DATA_QUBITS + 1  # 15
EDGE_BASE = NUM_DATA_QUBITS + NUM_ANCILLA_QUBITS  # 13

FIXED_EDGE_BELL_QUBITS = {}  # edge -> [14 Bell pair qubits]
FIXED_EDGE_PATH_QUBIT = {}   # edge -> 1 path qubit

for edge_idx, edge in enumerate(_edges):
    start = EDGE_BASE + edge_idx * EDGE_STRIDE
    FIXED_EDGE_BELL_QUBITS[edge] = list(range(start, start + 2 * NUM_DATA_QUBITS))
    FIXED_EDGE_PATH_QUBIT[edge] = start + 2 * NUM_DATA_QUBITS

TOTAL_FIXED_QUBITS = EDGE_BASE + len(_edges) * EDGE_STRIDE

print("=" * 60)
print("FIXED ALLOCATION MAPPING")
print("=" * 60)
print(f"Network: {len(network_graph.nodes())} nodes, {len(_edges)} edges")
print(f"")
print(f"Source data qubits:  {SOURCE_DATA_QUBITS}")
print(f"Sink ancilla qubits: {SINK_ANCILLA_QUBITS}")
print(f"")
print("Edge qubit assignments:")
for edge in _edges:
    bell = FIXED_EDGE_BELL_QUBITS[edge]
    path = FIXED_EDGE_PATH_QUBIT[edge]
    print(f"  Edge {edge}: Bell pairs {bell[0]}-{bell[-1]}, Path qubit {path}")
print(f"")
print(f"Total qubits: {TOTAL_FIXED_QUBITS}")
print("=" * 60)

## Noise Model

In [ ]:
# =============================================================================
# NOISE MODEL
# =============================================================================
# With fixed allocation, we build ONE noise model for all paths.
# - 1Q depolarizing errors from FakeFez gate properties
# - Custom 2Q depolarizing errors (all-to-all)

fake_backend = FakeFez()
target = fake_backend.target

def build_fixed_noise_model(total_qubits: int, error_rate_2q: float = 0.01) -> NoiseModel:
    """Build noise model for fixed allocation circuits."""
    noise_model = NoiseModel()
    
    # 1Q depolarizing errors from FakeFez
    gates_to_use = ['sx', 'x', 'rz', 'id']
    for gate in gates_to_use:
        if gate not in target.operation_names:
            continue
        for qargs in target.qargs_for_operation_name(gate):
            phys_qubit = qargs[0]
            if phys_qubit < total_qubits:
                props = target[gate][qargs]
                if props and props.error and props.error > 0:
                    depol_err = depolarizing_error(props.error, 1)
                    noise_model.add_quantum_error(depol_err, gate, [phys_qubit])
    
    # 2Q depolarizing errors (all-to-all, CZ only for FakeFez)
    error_2q = depolarizing_error(error_rate_2q, 2)
    for i in range(total_qubits):
        for j in range(i + 1, total_qubits):
            noise_model.add_quantum_error(error_2q, 'cz', [i, j])
            noise_model.add_quantum_error(error_2q, 'cz', [j, i])
    
    return noise_model

# Build the shared noise model
print(f"Building noise model for {TOTAL_FIXED_QUBITS} qubits...")
start = time.time()
NOISE_MODEL = build_fixed_noise_model(TOTAL_FIXED_QUBITS, ERROR_RATE_2Q)
print(f"Noise model built in {time.time() - start:.2f}s")
print(f"")
print(f"Backend: {fake_backend.name} ({fake_backend.num_qubits} qubits)")
print(f"Basis gates: {BASIS_GATES}")
print(f"2Q error rate: {ERROR_RATE_2Q}")

## [[7,1,3]] Steane Code Functions

In [ ]:
# =============================================================================
# [[7,1,3]] STEANE CODE FUNCTIONS
# =============================================================================

def apply_encoding_713(qc: QuantumCircuit, qubits: List[int]) -> None:
    """Apply [[7,1,3]] Steane code encoding circuit. Logical qubit is at qubits[0]."""
    if len(qubits) != 7:
        raise ValueError(f"Expected 7 qubits, got {len(qubits)}")

    q = qubits

    # Hadamard on qubits that will create X-stabilizer superposition
    qc.h(q[4])
    qc.h(q[5])
    qc.h(q[6])

    # CNOT propagation to create the code structure
    qc.cx(q[6], q[0])
    qc.cx(q[6], q[1])
    qc.cx(q[6], q[3])

    qc.cx(q[5], q[0])
    qc.cx(q[5], q[2])
    qc.cx(q[5], q[3])

    qc.cx(q[4], q[1])
    qc.cx(q[4], q[2])
    qc.cx(q[4], q[3])


def apply_syndrome_measurement_713(qc: QuantumCircuit, data_qubits: List[int],
                                    ancilla_qubits: List[int], classical_bits,
                                    reset_ancillas: bool = True) -> None:
    """Extract syndrome and measure ancilla qubits for [[7,1,3]] Steane code."""
    if len(data_qubits) != 7:
        raise ValueError(f"Expected 7 data qubits, got {len(data_qubits)}")
    if len(ancilla_qubits) != 6:
        raise ValueError(f"Expected 6 ancilla qubits, got {len(ancilla_qubits)}")

    if reset_ancillas:
        for ancilla in ancilla_qubits:
            qc.reset(ancilla)

    for idx, stabilizer in enumerate(STABILIZERS_713):
        ancilla = ancilla_qubits[idx]

        # Determine if this is an X-type or Z-type stabilizer
        first_non_identity = next((c for c in stabilizer if c != 'I'), 'I')
        is_x_type = (first_non_identity == 'X')

        if is_x_type:
            # X-type stabilizer: H-CNOT(ancilla->data)-H pattern
            qc.h(ancilla)
            for qubit_idx, pauli in enumerate(stabilizer):
                if pauli == 'X':
                    qc.cx(ancilla, data_qubits[qubit_idx])
            qc.h(ancilla)
        else:
            # Z-type stabilizer: CNOT(data->ancilla) pattern
            for qubit_idx, pauli in enumerate(stabilizer):
                if pauli == 'Z':
                    qc.cx(data_qubits[qubit_idx], ancilla)

        qc.measure(ancilla, classical_bits[idx])

## Entanglement Swapping Primitives

In [ ]:
# =============================================================================
# ENTANGLEMENT SWAPPING PRIMITIVES
# =============================================================================

def distribute_bell_pairs_sequential(qc, sender_qubits, receiver_qubits, path_qubit):
    """
    Distribute 7 Bell pairs from sender to receiver via physical SWAPs.
    
    Each of the 7 code qubits gets its own Bell pair:
        - sender_qubits[i] holds one half
        - receiver_qubits[i] holds the other half (after SWAP through channel)
    """
    for i in range(NUM_DATA_QUBITS):
        # Create Bell pair
        qc.h(sender_qubits[i])
        qc.cx(sender_qubits[i], path_qubit)
        # SWAP to receiver (noisy channel)
        qc.swap(path_qubit, receiver_qubits[i])


def apply_bell_measurement(qc, qubits_a, qubits_b, classical_bits):
    """
    Perform Bell measurements on pairs of qubits.
    
    Measurement outcomes encode required Pauli corrections:
        classical_bits[i]   (from qubit_a after H) -> Z correction if 1
        classical_bits[i+7] (from qubit_b)         -> X correction if 1
    """
    for i, (qa, qb) in enumerate(zip(qubits_a, qubits_b)):
        qc.cx(qa, qb)
        qc.h(qa)
        qc.measure(qa, classical_bits[i])
        qc.measure(qb, classical_bits[i + NUM_DATA_QUBITS])


def apply_pauli_corrections(qc, target_qubits, classical_register):
    """
    Apply Pauli corrections based on Bell measurement outcomes.
    
    - If classical_bits[i] = 1 -> apply Z to target[i]
    - If classical_bits[i+7] = 1 -> apply X to target[i]
    """
    for i in range(NUM_DATA_QUBITS):
        with qc.if_test((classical_register[i], 1)):
            qc.z(target_qubits[i])
        with qc.if_test((classical_register[i + NUM_DATA_QUBITS], 1)):
            qc.x(target_qubits[i])

## Fixed Path Allocation Class

In [ ]:
# =============================================================================
# FIXED PATH ALLOCATION
# =============================================================================

class FixedPathAllocation:
    """
    Fixed qubit allocation for consistent noise profiles.
    
    - Source data: ALWAYS indices 0-6
    - Sink ancilla: ALWAYS indices 7-12
    - Edge (i,j) Bell pairs: ALWAYS same fixed indices
    """
    
    def __init__(self, path):
        self.path = path
        self.source = path[0]
        self.sink = path[-1]
        self.edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
        
        self.source_data = SOURCE_DATA_QUBITS.copy()
        self.sink_ancilla = SINK_ANCILLA_QUBITS.copy()
        
        self.edge_qubits = {}
        self.path_qubit = {}
        
        for edge in self.edges:
            edge_key = get_edge_key(edge[0], edge[1])
            self.edge_qubits[edge_key] = FIXED_EDGE_BELL_QUBITS[edge_key].copy()
            self.path_qubit[edge_key] = FIXED_EDGE_PATH_QUBIT[edge_key]
        
        self.total_qubits = TOTAL_FIXED_QUBITS
    
    def get_node_edge_qubits(self, node, neighbor):
        """Get the 7 Bell pair qubits for this node's side of the edge."""
        edge_key = get_edge_key(node, neighbor)
        qubits = self.edge_qubits[edge_key]
        return qubits[0:NUM_DATA_QUBITS] if node < neighbor else qubits[NUM_DATA_QUBITS:]
    
    def get_path_qubit(self, node_a, node_b):
        """Get the path qubit for an edge."""
        edge_key = get_edge_key(node_a, node_b)
        return self.path_qubit[edge_key]

## Circuit Builder

In [ ]:
# =============================================================================
# CIRCUIT BUILDER
# =============================================================================

def build_es_circuit_fixed(path: list, initial_state: str = '0'):
    """
    Build [[7,1,3]] entanglement swapping circuit with fixed allocation.
    
    Protocol:
        1. Encode at source
        2. Distribute Bell pairs along path
        3. Entanglement swapping at intermediate nodes
        4. Teleport from source
        5. Apply Pauli corrections at sink
        6. Syndrome measurement at sink
    """
    alloc = FixedPathAllocation(path)
    
    source = path[0]
    sink = path[-1]
    intermediate_nodes = path[1:-1]
    edges = alloc.edges
    
    qc = QuantumCircuit(alloc.total_qubits)
    
    # Classical registers
    swap_registers = {}
    for node in intermediate_nodes:
        reg = ClassicalRegister(2 * NUM_DATA_QUBITS, name=f'c_swap_{node}')
        qc.add_register(reg)
        swap_registers[node] = reg
    
    teleport_register = ClassicalRegister(2 * NUM_DATA_QUBITS, name=f'c_teleport')
    qc.add_register(teleport_register)
    
    syndrome_register = ClassicalRegister(NUM_ANCILLA_QUBITS, name='c_syndrome')
    qc.add_register(syndrome_register)
    
    # Step 1: Encode at source
    if initial_state == '1':
        qc.x(alloc.source_data[LOGICAL_QUBIT_INDEX])
    apply_encoding_713(qc, alloc.source_data)
    qc.barrier(label='Encode')
    
    # Step 2: Bell pair distribution
    for edge in edges:
        sender, receiver = edge
        edge_key = get_edge_key(sender, receiver)
        
        if sender < receiver:
            sender_qubits = alloc.edge_qubits[edge_key][0:NUM_DATA_QUBITS]
            receiver_qubits = alloc.edge_qubits[edge_key][NUM_DATA_QUBITS:]
        else:
            sender_qubits = alloc.edge_qubits[edge_key][NUM_DATA_QUBITS:]
            receiver_qubits = alloc.edge_qubits[edge_key][0:NUM_DATA_QUBITS]
        
        path_qubit = alloc.path_qubit[edge_key]
        distribute_bell_pairs_sequential(qc, sender_qubits, receiver_qubits, path_qubit)
        qc.barrier(label=f'Bell {sender}->{receiver}')
    
    # Step 3: Entanglement swapping at intermediate nodes
    for i, node in enumerate(intermediate_nodes):
        prev_node = path[i]
        next_node = path[i + 2]
        incoming = alloc.get_node_edge_qubits(node, prev_node)
        outgoing = alloc.get_node_edge_qubits(node, next_node)
        apply_bell_measurement(qc, incoming, outgoing, swap_registers[node])
    
    if intermediate_nodes:
        qc.barrier(label='Swap')
    
    # Step 4: Teleportation from source
    source_link = alloc.get_node_edge_qubits(source, path[1])
    apply_bell_measurement(qc, alloc.source_data, source_link, teleport_register)
    qc.barrier(label='Teleport')
    
    # Step 5: Pauli corrections at sink
    sink_data = alloc.get_node_edge_qubits(sink, path[-2])
    for node in intermediate_nodes:
        apply_pauli_corrections(qc, sink_data, swap_registers[node])
    apply_pauli_corrections(qc, sink_data, teleport_register)
    qc.barrier(label='Corrections')
    
    # Step 6: Syndrome measurement
    apply_syndrome_measurement_713(qc, sink_data, alloc.sink_ancilla, syndrome_register)
    
    return qc, alloc

## Data Container

In [ ]:
# =============================================================================
# DATA CONTAINER
# =============================================================================

class TimeAwareMeasurement:
    def __init__(self, path_edges, histogram, duration, latency_stats=None, code_type='713_ES'):
        self.path_edges = path_edges
        self.histogram = histogram
        self.duration = duration
        self.latency_stats = latency_stats
        self.code_type = code_type

## Test Circuit

In [ ]:
# =============================================================================
# TEST CIRCUIT
# =============================================================================

test_path = [0, 1, 2]
test_circuit, test_alloc = build_es_circuit_fixed(test_path, initial_state='0')

print(f"Path: {test_path}")
print(f"Qubits: {test_circuit.num_qubits} (FIXED)")
print(f"Classical bits: {test_circuit.num_clbits}")
print(f"Gate counts: {test_circuit.count_ops()}")

In [ ]:
# Noiseless test
simulator = AerSimulator(method='matrix_product_state', seed_simulator=SEED)
transpiled = transpile(test_circuit, basis_gates=BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL)

result = simulator.run(transpiled, shots=NUM_SHOTS, seed_simulator=SEED).result()

# Extract syndrome
from qiskit.result import marginal_distribution
syndrome_reg = test_circuit.cregs[-1]
syndrome_indices = [test_circuit.find_bit(bit).index for bit in syndrome_reg]
syndrome_counts = marginal_distribution(result.get_counts(), indices=syndrome_indices)

print(f"Noiseless syndrome counts: {dict(sorted(syndrome_counts.items()))}")
plot_histogram(syndrome_counts)

In [ ]:
# Noisy test
noisy_sim = AerSimulator(noise_model=NOISE_MODEL, method='matrix_product_state', seed_simulator=SEED)
noisy_result = noisy_sim.run(transpiled, shots=NUM_SHOTS, seed_simulator=SEED).result()

noisy_counts = marginal_distribution(noisy_result.get_counts(), indices=syndrome_indices)
print(f"Noisy syndrome counts: {dict(sorted(noisy_counts.items()))}")

no_error_frac = noisy_counts.get('000000', 0) / sum(noisy_counts.values())
print(f"No-error fraction: {no_error_frac:.4f}")
plot_histogram(noisy_counts)

## Data Generation

In [ ]:
# Generate all paths
ALL_PATHS = []
for pair in combinations(network_graph.nodes, 2):
    paths = nx.all_simple_paths(network_graph, source=pair[0], target=pair[1], cutoff=MAX_HOPS)
    ALL_PATHS.extend([p for p in paths if len(p) > MIN_HOPS])

print(f"Total paths ({MIN_HOPS}-{MAX_HOPS} hops): {len(ALL_PATHS)}")
print(f"Sample: {ALL_PATHS[:3]}")

In [ ]:
# =============================================================================
# DATA GENERATION
# =============================================================================

measurements = []
noisy_sim = AerSimulator(noise_model=NOISE_MODEL, method='matrix_product_state', seed_simulator=SEED)

print("=" * 70)
print("[[7,1,3]] STEANE CODE - ENTANGLEMENT SWAPPING - FIXED ALLOCATION")
print("=" * 70)
print(f"Total qubits: {TOTAL_FIXED_QUBITS}")
print(f"2Q error rate: {ERROR_RATE_2Q}")
print(f"Paths: {len(ALL_PATHS)}")
print("=" * 70)

total_start = time.time()

for idx, path in enumerate(ALL_PATHS):
    path_start = time.time()
    
    circuit, alloc = build_es_circuit_fixed(path, initial_state='0')
    transpiled = transpile(circuit, basis_gates=BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL, seed_transpiler=SEED)
    
    result = noisy_sim.run(transpiled, shots=NUM_SHOTS, seed_simulator=SEED + idx).result()
    
    syndrome_reg = circuit.cregs[-1]
    syndrome_indices = [circuit.find_bit(bit).index for bit in syndrome_reg]
    syndrome_counts = marginal_distribution(result.get_counts(), indices=syndrome_indices)
    
    histogram = np.array([syndrome_counts.get(f"{i:06b}", 0) for i in range(NUM_SYNDROMES)], dtype=np.float32)
    
    path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
    elapsed = time.time() - path_start
    
    measurements.append(TimeAwareMeasurement(
        path_edges=path_edges,
        histogram=histogram,
        duration=elapsed,
        latency_stats={'shots': NUM_SHOTS, 'qubits': TOTAL_FIXED_QUBITS},
        code_type='713_ES'
    ))
    
    no_error_frac = histogram[0] / histogram.sum()
    edges_str = "->".join(str(n) for n in path)
    print(f"[{idx+1:2d}/{len(ALL_PATHS)}] {edges_str:15s} | {elapsed:5.1f}s | no-error: {no_error_frac:.3f}")

total_elapsed = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"Total: {len(measurements)} measurements in {total_elapsed:.1f}s")
print(f"Average: {total_elapsed/len(ALL_PATHS):.1f}s per path")
print(f"{'=' * 70}")

## Ground Truth Generation

In [ ]:
# =============================================================================
# GROUND TRUTH CIRCUIT
# =============================================================================

def build_ground_truth_circuit_es(path, initial_state='0', measurement_basis='Z'):
    """Build ground truth circuit for single-qubit transport via entanglement swapping."""
    alloc = FixedPathAllocation(path)
    
    source = path[0]
    sink = path[-1]
    intermediate_nodes = path[1:-1]
    
    qc = QuantumCircuit(alloc.total_qubits)
    
    swap_registers = {}
    for node in intermediate_nodes:
        reg = ClassicalRegister(2, name=f'c_swap_{node}')
        qc.add_register(reg)
        swap_registers[node] = reg
    
    teleport_register = ClassicalRegister(2, name='c_teleport')
    qc.add_register(teleport_register)
    
    measure_register = ClassicalRegister(1, name='c_measure')
    qc.add_register(measure_register)
    
    source_qubit = alloc.source_data[0]
    
    if initial_state == '1':
        qc.x(source_qubit)
    if measurement_basis == 'X':
        qc.h(source_qubit)
    elif measurement_basis == 'Y':
        qc.h(source_qubit)
        qc.s(source_qubit)
    
    # Bell pair distribution (single qubit)
    for edge in alloc.edges:
        sender, receiver = edge
        edge_key = get_edge_key(sender, receiver)
        
        if sender < receiver:
            sender_qubit = alloc.edge_qubits[edge_key][0]
            receiver_qubit = alloc.edge_qubits[edge_key][NUM_DATA_QUBITS]
        else:
            sender_qubit = alloc.edge_qubits[edge_key][NUM_DATA_QUBITS]
            receiver_qubit = alloc.edge_qubits[edge_key][0]
        
        path_qubit = alloc.path_qubit[edge_key]
        qc.h(sender_qubit)
        qc.cx(sender_qubit, path_qubit)
        qc.swap(path_qubit, receiver_qubit)
    
    # Entanglement swapping (single qubit)
    def get_single_qubit(node, neighbor):
        edge_key = get_edge_key(node, neighbor)
        return alloc.edge_qubits[edge_key][0] if node < neighbor else alloc.edge_qubits[edge_key][NUM_DATA_QUBITS]
    
    for i, node in enumerate(intermediate_nodes):
        incoming = get_single_qubit(node, path[i])
        outgoing = get_single_qubit(node, path[i + 2])
        qc.cx(incoming, outgoing)
        qc.h(incoming)
        qc.measure(incoming, swap_registers[node][0])
        qc.measure(outgoing, swap_registers[node][1])
    
    # Teleportation
    source_link = get_single_qubit(source, path[1])
    qc.cx(source_qubit, source_link)
    qc.h(source_qubit)
    qc.measure(source_qubit, teleport_register[0])
    qc.measure(source_link, teleport_register[1])
    
    # Corrections
    sink_qubit = get_single_qubit(sink, path[-2])
    for node in intermediate_nodes:
        with qc.if_test((swap_registers[node][0], 1)):
            qc.z(sink_qubit)
        with qc.if_test((swap_registers[node][1], 1)):
            qc.x(sink_qubit)
    with qc.if_test((teleport_register[0], 1)):
        qc.z(sink_qubit)
    with qc.if_test((teleport_register[1], 1)):
        qc.x(sink_qubit)
    
    # Measure
    if measurement_basis == 'X':
        qc.h(sink_qubit)
    elif measurement_basis == 'Y':
        qc.sdg(sink_qubit)
        qc.h(sink_qubit)
    qc.measure(sink_qubit, measure_register[0])
    
    return qc, alloc


def calculate_pauli_rates(z_counts, x_counts, y_counts, expected='0'):
    """Calculate Pauli error rates from three-basis measurements."""
    error = '1' if expected == '0' else '0'
    E_z = z_counts.get(error, 0) / sum(z_counts.values())
    E_x = x_counts.get(error, 0) / sum(x_counts.values())
    E_y = y_counts.get(error, 0) / sum(y_counts.values())
    
    p_x = max(0, (E_z + E_y - E_x) / 2)
    p_y = max(0, (E_z + E_x - E_y) / 2)
    p_z = max(0, (E_x + E_y - E_z) / 2)
    
    return {'p_x': p_x, 'p_y': p_y, 'p_z': p_z, 'E_x': E_x, 'E_y': E_y, 'E_z': E_z}

In [ ]:
# =============================================================================
# GROUND TRUTH DATA GENERATION
# =============================================================================

GROUND_TRUTH_DATA = {}

print("=" * 70)
print("GROUND TRUTH DATA GENERATION")
print("=" * 70)
print(f"Edges: {list(network_graph.edges())}")
print("=" * 70)

for edge_idx, edge in enumerate(network_graph.edges()):
    path = [edge[0], edge[1]]
    print(f"\nEdge {edge}:")
    
    cZ, _ = build_ground_truth_circuit_es(path, measurement_basis='Z')
    cX, _ = build_ground_truth_circuit_es(path, measurement_basis='X')
    cY, _ = build_ground_truth_circuit_es(path, measurement_basis='Y')
    
    tZ = transpile(cZ, basis_gates=BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL, seed_transpiler=SEED)
    tX = transpile(cX, basis_gates=BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL, seed_transpiler=SEED)
    tY = transpile(cY, basis_gates=BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL, seed_transpiler=SEED)
    
    print("  Z-basis...", end=' | ')
    rZ = noisy_sim.run(tZ, shots=NUM_SHOTS, seed_simulator=SEED + edge_idx * 3).result()
    idx_Z = [cZ.find_bit(bit).index for bit in cZ.cregs[-1]]
    counts_Z = marginal_distribution(rZ.get_counts(), indices=idx_Z)
    
    print("X-basis...", end=' | ')
    rX = noisy_sim.run(tX, shots=NUM_SHOTS, seed_simulator=SEED + edge_idx * 3 + 1).result()
    idx_X = [cX.find_bit(bit).index for bit in cX.cregs[-1]]
    counts_X = marginal_distribution(rX.get_counts(), indices=idx_X)
    
    print("Y-basis...")
    rY = noisy_sim.run(tY, shots=NUM_SHOTS, seed_simulator=SEED + edge_idx * 3 + 2).result()
    idx_Y = [cY.find_bit(bit).index for bit in cY.cregs[-1]]
    counts_Y = marginal_distribution(rY.get_counts(), indices=idx_Y)
    
    rates = calculate_pauli_rates(counts_Z, counts_X, counts_Y)
    GROUND_TRUTH_DATA[edge] = [rates['p_x'], rates['p_y'], rates['p_z']]
    print(f"  PX={rates['p_x']:.6f}, PY={rates['p_y']:.6f}, PZ={rates['p_z']:.6f}")

print(f"\n{'=' * 70}")
print("Ground truth complete")
print(f"{'=' * 70}")

## Save Data

In [ ]:
# =============================================================================
# SAVE DATA
# =============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, 'measurements.pkl'), 'wb') as f:
    pickle.dump(measurements, f)

with open(os.path.join(OUTPUT_DIR, 'graph.pkl'), 'wb') as f:
    pickle.dump(network_graph, f)

with open(os.path.join(OUTPUT_DIR, 'ground_truth.pkl'), 'wb') as f:
    pickle.dump(GROUND_TRUTH_DATA, f)

print(f"Data saved to {OUTPUT_DIR}")
print(f"  - measurements.pkl ({len(measurements)} measurements)")
print(f"  - graph.pkl")
print(f"  - ground_truth.pkl ({len(GROUND_TRUTH_DATA)} edges)")
print(f"\nGround truth summary:")
for edge, rates in GROUND_TRUTH_DATA.items():
    print(f"  Edge {edge}: PX={rates[0]:.6f}, PY={rates[1]:.6f}, PZ={rates[2]:.6f}")

## Summary

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 60)
print("ANALYSIS SUMMARY")
print("=" * 60)

no_error_fractions = [(m.path_edges, m.histogram[0] / m.histogram.sum()) for m in measurements]
no_error_fractions.sort(key=lambda x: x[1])

print("\nLowest no-error fraction paths:")
for edges, frac in no_error_fractions[:5]:
    print(f"  {edges}: {frac:.4f}")

print("\nHighest no-error fraction paths:")
for edges, frac in no_error_fractions[-5:]:
    print(f"  {edges}: {frac:.4f}")

avg_no_error = np.mean([f for _, f in no_error_fractions])
print(f"\nAverage no-error fraction: {avg_no_error:.4f}")